## 연습 1: Solaris Energy 데이터 플랫폼 설정

데이터를 수집하기 전에 Unity Catalog 구조를 설정해야 합니다: **플랫폼을 나타내는 카탈로그**, **처리 계층별로 데이터를 구성하는 스키마**, 그리고 **원본 파일을 저장하는 볼륨**.

이 연습을 완료한 후, 랩 설정 페이지로 돌아가서 **Catalog Explorer**에서 생성한 객체를 확인합니다.

### 작업 1: Unity Catalog 구조 설정

이전 랩에서 카탈로그, 스키마 및 볼륨 생성을 이미 연습했으므로, 이 설정 코드는 제공됩니다 — 단순히 아래 셀을 실행하여 환경을 준비합니다.

코드는 다음 Unity Catalog 객체를 생성합니다:
- **카탈로그** `solaris_lab` — Solaris Energy 플랫폼의 최상위 네임스페이스
- **bronze 스키마** `solaris_lab` 내부 — 원본 시스템에서 정확히 도착한 원본 수집 데이터용
- **silver 스키마** `solaris_lab` 내부 — 정제되고 풍부한 데이터용
- **managed 볼륨** `raw_files` `solaris_lab.bronze` 내부 — 현장 센서 및 SCADA 시스템의 수신 CSV 파일용 착륙 영역

In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS solaris_lab
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Solaris Energy analytics platform — solar farms and wind turbines across Europe'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS solaris_lab.bronze
  COMMENT 'Raw ingested data — unmodified, as received from field systems';

CREATE SCHEMA IF NOT EXISTS solaris_lab.silver
  COMMENT 'Cleansed and enriched data — ready for analytics and reporting';
  
CREATE VOLUME IF NOT EXISTS solaris_lab.bronze.raw_files
  COMMENT 'Landing zone for raw CSV files from SCADA systems and field sensors';

### ✅ 제공됨: 볼륨을 샘플 데이터로 채우기

아래 셀을 실행하여 2개의 샘플 CSV 파일을 볼륨에 작성합니다:
- `solar_readings.csv` — 스페인과 독일의 사이트에서 태양광 패널의 에너지 출력 판독값
- `turbine_events.csv` — 네덜란드와 덴마크의 풍력 터빈의 유지보수 및 경고 이벤트

연습 2, 3, 4에서 이 파일들을 사용합니다.

In [ ]:
# ✅ Run this cell — it writes sample data files to your volume

solar_csv = """reading_id,site_id,panel_id,reading_timestamp,power_kw,irradiance_w_m2,temperature_c,status
R001,SITE_ES_001,P-ES-001,2025-06-01 06:30:00,8.2,210,18.4,Normal
R002,SITE_ES_001,P-ES-002,2025-06-01 06:30:00,7.9,205,18.6,Normal
R003,SITE_ES_001,P-ES-003,2025-06-01 06:30:00,8.5,215,18.2,Normal
R004,SITE_ES_001,P-ES-004,2025-06-01 06:30:00,0.0,205,18.5,Fault
R005,SITE_ES_001,P-ES-005,2025-06-01 06:30:00,7.6,200,18.8,Normal
R006,SITE_ES_001,P-ES-001,2025-06-01 09:00:00,142.3,750,28.1,Normal
R007,SITE_ES_001,P-ES-002,2025-06-01 09:00:00,138.7,745,28.4,Normal
R008,SITE_ES_001,P-ES-003,2025-06-01 09:00:00,145.1,755,27.9,Normal
R009,SITE_ES_001,P-ES-004,2025-06-01 09:00:00,0.0,750,28.2,Fault
R010,SITE_ES_001,P-ES-005,2025-06-01 09:00:00,136.4,740,28.7,Normal
R011,SITE_ES_001,P-ES-001,2025-06-01 12:00:00,228.5,1150,41.3,Normal
R012,SITE_ES_001,P-ES-002,2025-06-01 12:00:00,221.4,1140,41.8,Warning
R013,SITE_ES_001,P-ES-003,2025-06-01 12:00:00,235.8,1160,40.9,Normal
R014,SITE_ES_001,P-ES-004,2025-06-01 12:00:00,0.0,1150,41.5,Fault
R015,SITE_ES_001,P-ES-005,2025-06-01 12:00:00,219.6,1135,42.1,Warning
R016,SITE_DE_001,P-DE-001,2025-06-01 06:30:00,4.1,120,12.3,Normal
R017,SITE_DE_001,P-DE-002,2025-06-01 06:30:00,4.3,125,12.1,Normal
R018,SITE_DE_001,P-DE-003,2025-06-01 06:30:00,3.8,118,12.5,Normal
R019,SITE_DE_001,P-DE-004,2025-06-01 06:30:00,4.0,122,12.4,Normal
R020,SITE_DE_001,P-DE-005,2025-06-01 06:30:00,4.2,124,12.2,Normal
R021,SITE_DE_001,P-DE-001,2025-06-01 09:00:00,98.7,540,19.8,Normal
R022,SITE_DE_001,P-DE-002,2025-06-01 09:00:00,101.2,548,19.6,Normal
R023,SITE_DE_001,P-DE-003,2025-06-01 09:00:00,96.4,535,20.1,Normal
R024,SITE_DE_001,P-DE-004,2025-06-01 09:00:00,99.8,542,19.9,Normal
R025,SITE_DE_001,P-DE-005,2025-06-01 09:00:00,102.1,550,19.4,Normal
R026,SITE_DE_001,P-DE-001,2025-06-01 12:00:00,187.3,980,31.2,Normal
R027,SITE_DE_001,P-DE-002,2025-06-01 12:00:00,191.6,992,30.8,Normal
R028,SITE_DE_001,P-DE-003,2025-06-01 12:00:00,184.9,975,31.5,Normal
R029,SITE_DE_001,P-DE-004,2025-06-01 12:00:00,188.7,984,31.0,Normal
R030,SITE_DE_001,P-DE-005,2025-06-01 12:00:00,193.2,996,30.6,Normal
R031,SITE_ES_001,P-ES-001,2025-06-02 12:00:00,232.1,1165,42.7,Normal
R032,SITE_ES_001,P-ES-002,2025-06-02 12:00:00,226.9,1155,43.1,Normal
R033,SITE_ES_001,P-ES-003,2025-06-02 12:00:00,239.4,1175,42.3,Normal
R034,SITE_ES_001,P-ES-004,2025-06-02 12:00:00,225.2,1148,43.4,Normal
R035,SITE_ES_001,P-ES-005,2025-06-02 12:00:00,231.7,1162,42.9,Normal
"""

turbine_csv = """event_id,turbine_id,site_id,event_timestamp,event_type,severity,description,resolved
EVT001,T-NL-001,WIND_NL_001,2025-06-01 02:14:00,Alert,Medium,Vibration level 18% above baseline threshold,true
EVT002,T-NL-002,WIND_NL_001,2025-06-01 03:47:00,Fault,High,Gearbox oil temperature exceeded 85 degrees C,true
EVT003,T-NL-003,WIND_NL_001,2025-06-01 07:30:00,Inspection,Low,Scheduled 6-month rotor blade inspection,true
EVT004,T-NL-004,WIND_NL_001,2025-06-01 09:12:00,Alert,Low,Wind speed sensor calibration drift detected,false
EVT005,T-NL-005,WIND_NL_001,2025-06-01 11:55:00,Maintenance,Medium,Yaw motor lubrication service,true
EVT006,T-DK-001,WIND_DK_001,2025-06-01 01:05:00,Alert,Critical,Blade pitch control failure - emergency stop activated,true
EVT007,T-DK-002,WIND_DK_001,2025-06-01 04:22:00,Fault,Medium,Power converter overload - output capped at 60%,false
EVT008,T-DK-003,WIND_DK_001,2025-06-01 06:45:00,Inspection,Low,Annual safety and compliance inspection,true
EVT009,T-DK-004,WIND_DK_001,2025-06-01 10:33:00,Maintenance,Low,Tower bolt torque check completed,true
EVT010,T-DK-005,WIND_DK_001,2025-06-01 14:18:00,Alert,Medium,Ice accretion on blades - de-icing initiated,true
EVT011,T-NL-001,WIND_NL_001,2025-06-02 05:30:00,Maintenance,Low,Hydraulic fluid top-up completed,true
EVT012,T-NL-002,WIND_NL_001,2025-06-02 08:00:00,Alert,High,Generator winding temperature 12 degrees C above limit,false
EVT013,T-NL-003,WIND_NL_001,2025-06-02 09:45:00,Fault,Medium,Anemometer replaced after sensor malfunction,true
EVT014,T-NL-004,WIND_NL_001,2025-06-02 12:10:00,Inspection,Low,Control system firmware update applied,true
EVT015,T-NL-005,WIND_NL_001,2025-06-02 15:22:00,Alert,Low,Brake pad wear indicator triggered,false
EVT016,T-DK-001,WIND_DK_001,2025-06-02 03:18:00,Maintenance,Medium,Main bearing grease replenishment,true
EVT017,T-DK-002,WIND_DK_001,2025-06-02 07:55:00,Alert,High,Pitch battery backup low - replacement scheduled,false
EVT018,T-DK-003,WIND_DK_001,2025-06-02 10:40:00,Fault,Critical,Transformer fault - turbine offline for 4 hours,true
EVT019,T-DK-004,WIND_DK_001,2025-06-02 13:15:00,Inspection,Low,Cooling system filters cleaned,true
EVT020,T-DK-005,WIND_DK_001,2025-06-02 16:50:00,Maintenance,Medium,Slip ring inspection and cleaning completed,true
"""

with open("/Volumes/solaris_lab/bronze/raw_files/solar_readings.csv", "w") as f:
    f.write(solar_csv.strip())

with open("/Volumes/solaris_lab/bronze/raw_files/turbine_events.csv", "w") as f:
    f.write(turbine_csv.strip())

print("✅ Sample data written to /Volumes/solaris_lab/bronze/raw_files/")

## 연습 2: PySpark DataFrame을 이용한 배치 수집

Solaris Energy의 SCADA 시스템은 일일 태양광 패널 판독값을 CSV 파일로 내보냅니다. 첫 번째 수집 작업은 최신 파일을 볼륨에서 읽고, 내용을 검사하고, `bronze` 계층에 managed 델타 테이블로 작성하는 것입니다.

이 연습을 완료한 후, 랩 설정 페이지로 돌아가서 **Catalog Explorer**에서 테이블을 검사합니다.

### 작업 2.1 — 태양광 판독값 CSV를 DataFrame으로 읽기

`spark.read` API를 사용하여 볼륨에서 `solar_readings.csv`를 로드합니다. 파일에는 헤더 행이 있으며 Spark가 열 데이터 타입을 자동으로 추론하도록 해야 합니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I read a CSV file from a Unity Catalog volume into a PySpark DataFrame with header and schema inference enabled?"*

**힌트:**
- 볼륨 경로는 `/Volumes/solaris_lab/bronze/raw_files/solar_readings.csv`입니다
- `.format("csv")`, `.option("header", "true")`, `.option("inferSchema", "true")`, `.load(...)`를 사용합니다

In [ ]:
df_solar = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/solaris_lab/bronze/raw_files/solar_readings.csv")
)

### 작업 2.2 — 스키마 및 샘플 행 검사

데이터를 테이블에 작성하기 전에 추론된 스키마를 검증하고 기록 샘플을 확인하는 것이 좋습니다. `power_kw`, `irradiance_w_m2`, `temperature_c`가 숫자 타입인지 확인합니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I print the schema of a PySpark DataFrame and display the first few rows?"*

**힌트:**
- `df_solar.printSchema()`를 사용하여 열 이름과 데이터 타입을 봅니다
- `display(df_solar)`를 사용하여 데이터 샘플을 미리봅니다

In [ ]:
# Print the schema — confirm power_kw, irradiance_w_m2, temperature_c are numeric
df_solar.printSchema()

# Display a sample of the data
display(df_solar)

### 작업 2.3 — DataFrame을 델타 테이블에 작성

태양광 판독값을 `solaris_lab.bronze.solar_readings`의 managed 델타 테이블로 유지합니다. Azure Databricks는 기본적으로 ACID 거래 및 시간 여행을 제공하는 Delta Lake 형식을 사용합니다.

셀이 실행될 때마다 테이블이 완전히 대체되도록 `overwrite` 모드를 사용합니다 (초기 완전 로드에 적합).

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I write a PySpark DataFrame to a Unity Catalog Delta table using saveAsTable in overwrite mode?"*

**힌트:**
- `df_solar.write.mode("overwrite").saveAsTable("solaris_lab.bronze.solar_readings")`를 사용합니다

In [ ]:
df_solar.write.mode("overwrite").saveAsTable("solaris_lab.bronze.solar_readings")

print("✅ Solar readings written to solaris_lab.bronze.solar_readings")

### 작업 2.4 — 수집된 테이블 검증

방금 생성한 델타 테이블을 조회하여 모든 행이 로드되었는지 확인합니다. 또한 `status = 'Fault'`인 행을 필터링하여 패널 `P-ES-004`의 오류 판독값이 있는지 확인합니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I count rows in a Databricks SQL table and filter by a column value?"*

**힌트:**
- `SELECT COUNT(*) FROM solaris_lab.bronze.solar_readings`를 사용합니다
- `WHERE status = 'Fault'` 절을 추가하여 오류를 필터링합니다

In [ ]:
%sql
-- Total row count — should be 35
SELECT COUNT(*) AS total_readings FROM solaris_lab.bronze.solar_readings;

In [ ]:
%sql
-- Fault readings — should be 3 (R004, R009, R014 from P-ES-004)
SELECT reading_id, panel_id, reading_timestamp, power_kw, status
FROM solaris_lab.bronze.solar_readings
WHERE status = 'Fault'
ORDER BY reading_timestamp;

## 연습 3: SQL 기반 수집

운영 팀은 풍력 터빈 이벤트 로그를 CSV 파일로 제공하며, 이는 시간에 따라 볼륨에 누적됩니다. 매 번 전체 파일을 다시 로드하는 대신 `COPY INTO`를 사용하여 멱등성이 있는 증분 로딩을 수행합니다. 그런 다음 `CREATE TABLE AS SELECT`를 사용하여 silver 계층 요약 테이블을 구축합니다.

### 작업 3.1 — 터빈 이벤트 대상 테이블 생성

`COPY INTO`를 실행하기 전에 대상 테이블이 존재해야 합니다. 다음 열을 포함하는 델타 테이블 `solaris_lab.bronze.turbine_events`를 생성합니다:

| 열 | 타입 |
|---|---|
| `event_id` | `STRING` |
| `turbine_id` | `STRING` |
| `site_id` | `STRING` |
| `event_timestamp` | `TIMESTAMP` |
| `event_type` | `STRING` |
| `severity` | `STRING` |
| `description` | `STRING` |
| `resolved` | `BOOLEAN` |

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I create a Delta table in Databricks SQL with explicit column types, using CREATE TABLE IF NOT EXISTS?"*

**힌트:**
- `CREATE TABLE IF NOT EXISTS solaris_lab.bronze.turbine_events (...)`를 사용합니다
- `USING DELTA`를 사용합니다 (Azure Databricks의 권장 저장소 형식)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS solaris_lab.bronze.turbine_events (
  event_id        STRING,
  turbine_id      STRING,
  site_id         STRING,
  event_timestamp TIMESTAMP,
  event_type      STRING,
  severity        STRING,
  description     STRING,
  resolved        BOOLEAN
)
USING DELTA
COMMENT 'Maintenance and alert events from wind turbines at Solaris Energy sites';

### 작업 3.2 — COPY INTO를 사용한 터빈 이벤트 로드

`COPY INTO`를 사용하여 볼륨에서 `turbine_events.csv`를 로드합니다. `COPY INTO`는 멱등성이 있습니다 — 이미 로드된 파일을 추적하므로 명령을 다시 실행해도 중복 행이 생성되지 않습니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I use COPY INTO in Databricks SQL to load a CSV file from a Unity Catalog volume, including FORMAT_OPTIONS for header and type inference?"*

**힌트:**
- `FROM` 절이 `/Volumes/solaris_lab/bronze/raw_files/turbine_events.csv`를 가리킵니다
- `FILEFORMAT = CSV`를 설정합니다
- `FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')`를 사용합니다

In [ ]:
%sql
COPY INTO solaris_lab.bronze.turbine_events
FROM '/Volumes/solaris_lab/bronze/raw_files/turbine_events.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true',
  'inferSchema' = 'true'
);

### 작업 3.3 — 멱등성 검증

`turbine_events`의 행을 개수한 다음, `COPY INTO`를 다시 실행하고, 두 번째 시간에 행을 다시 개수합니다. 두 개수가 동일해야 합니다 — `COPY INTO`가 이미 로드된 파일을 건너뛴다는 것을 보여줍니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How does COPY INTO track which files have already been loaded in Databricks?"*

**힌트:**
- `COPY INTO` 재실행 전후에 `SELECT COUNT(*) FROM solaris_lab.bronze.turbine_events`를 사용합니다
- 두 개수는 동일한 번호를 반환해야 합니다

In [ ]:
%sql
-- First count — should be 20
SELECT COUNT(*) AS event_count FROM solaris_lab.bronze.turbine_events;

In [ ]:
%sql
-- Re-run COPY INTO (same command as Task 3.2)
COPY INTO solaris_lab.bronze.turbine_events
FROM '/Volumes/solaris_lab/bronze/raw_files/turbine_events.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true',
  'inferSchema' = 'true'
);

In [ ]:
%sql
-- Second count — should still be 20 (no duplicates)
SELECT COUNT(*) AS event_count FROM solaris_lab.bronze.turbine_events;

### 작업 3.4 — CTAS를 사용한 silver 요약 테이블 구축

에너지 분석 팀은 각 사이트의 출력을 추적하기 위한 일일 생산 요약이 필요합니다. `CREATE TABLE AS SELECT` (`CTAS`)를 사용하여 bronze `solar_readings` 테이블로부터 집계하여 `solaris_lab.silver.daily_solar_production`을 생성합니다.

요약은 다음을 포함해야 합니다:
- `reading_date` — `reading_timestamp`의 날짜 부분
- `site_id`
- `total_power_kw` — Normal 상태 판독값만 `power_kw` 합계
- `avg_irradiance` — `irradiance_w_m2` 평균
- `fault_count` — `status = 'Fault'`인 판독값 개수

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I use CREATE TABLE AS SELECT in Databricks SQL to aggregate data from an existing table, grouping by date and site?"*

**힌트:**
- `CREATE OR REPLACE TABLE solaris_lab.silver.daily_solar_production AS SELECT ...`를 사용합니다
- 날짜를 추출하기 위해 `CAST(reading_timestamp AS DATE)` 또는 `DATE(reading_timestamp)`를 사용합니다
- 조건부 집계를 위해 `SUM(CASE WHEN status = 'Normal' THEN power_kw ELSE 0 END)`를 사용합니다
- `reading_date`와 `site_id`별로 그룹화합니다

In [ ]:
%sql
CREATE OR REPLACE TABLE solaris_lab.silver.daily_solar_production AS
SELECT
  DATE(reading_timestamp)                                              AS reading_date,
  site_id,
  ROUND(SUM(CASE WHEN status = 'Normal' THEN power_kw ELSE 0 END), 2) AS total_power_kw,
  ROUND(AVG(irradiance_w_m2), 1)                                      AS avg_irradiance,
  COUNT(CASE WHEN status = 'Fault' THEN 1 END)                        AS fault_count
FROM solaris_lab.bronze.solar_readings
GROUP BY reading_date, site_id
ORDER BY reading_date, site_id;

In [ ]:
%sql
SELECT * FROM solaris_lab.silver.daily_solar_production
ORDER BY reading_date, site_id;

## 연습 4: Auto Loader를 이용한 연속 파일 수집

새로운 태양광 판독값은 15분마다 SCADA 시스템에서 도착하여 클라우드 스토리지 디렉토리에 개별 CSV 파일로 도착합니다. 새 파일을 검색하기 위해 배치 작업을 예약하는 대신 **Auto Loader**를 사용하여 자동으로 감지하고 수집합니다 — 각 파일을 정확히 한 번 처리합니다.

Auto Loader는 `spark.readStream`과 함께 `cloudFiles` 소스 형식을 사용합니다. `trigger(availableNow=True)` 옵션은 현재 사용 가능한 모든 파일을 처리한 다음 중지하며, 이는 예약된 또는 주문형 수집에 이상적입니다.

### ✅ 제공됨: 수신 데이터 파일 시뮬레이션

아래 셀을 실행합니다. 3개의 분리된 CSV 파일을 볼륨의 `incoming/` 하위 디렉토리에 생성하여 3개의 연속 SCADA 내보내기 웨이브를 시뮬레이션합니다. Auto Loader가 이들을 모두 감지하고 처리합니다.

In [ ]:
# ✅ Run this cell — it writes three incoming data files

import os

incoming_dir = "/Volumes/solaris_lab/bronze/raw_files/incoming"
os.makedirs(incoming_dir, exist_ok=True)

wave1 = """reading_id,site_id,panel_id,reading_timestamp,power_kw,irradiance_w_m2,temperature_c,status
R036,SITE_DE_001,P-DE-001,2025-06-03 09:00:00,103.4,558,20.3,Normal
R037,SITE_DE_001,P-DE-002,2025-06-03 09:00:00,99.7,545,20.6,Normal
R038,SITE_DE_001,P-DE-003,2025-06-03 09:00:00,105.1,562,20.1,Normal
R039,SITE_DE_001,P-DE-004,2025-06-03 09:00:00,101.3,552,20.4,Warning
R040,SITE_DE_001,P-DE-005,2025-06-03 09:00:00,97.8,540,20.8,Normal"""

wave2 = """reading_id,site_id,panel_id,reading_timestamp,power_kw,irradiance_w_m2,temperature_c,status
R041,SITE_ES_001,P-ES-001,2025-06-03 09:15:00,148.2,765,29.4,Normal
R042,SITE_ES_001,P-ES-002,2025-06-03 09:15:00,143.5,755,29.8,Normal
R043,SITE_ES_001,P-ES-003,2025-06-03 09:15:00,151.9,778,29.1,Normal
R044,SITE_ES_001,P-ES-004,2025-06-03 09:15:00,146.7,761,29.5,Normal
R045,SITE_ES_001,P-ES-005,2025-06-03 09:15:00,140.3,748,30.1,Normal"""

wave3 = """reading_id,site_id,panel_id,reading_timestamp,power_kw,irradiance_w_m2,temperature_c,status
R046,SITE_DE_001,P-DE-001,2025-06-03 09:30:00,108.6,570,20.9,Normal
R047,SITE_DE_001,P-DE-002,2025-06-03 09:30:00,104.9,562,21.2,Normal
R048,SITE_DE_001,P-DE-003,2025-06-03 09:30:00,110.3,575,20.7,Normal
R049,SITE_DE_001,P-DE-004,2025-06-03 09:30:00,0.0,565,21.0,Fault
R050,SITE_DE_001,P-DE-005,2025-06-03 09:30:00,102.7,558,21.4,Normal"""

for name, content in [("wave_1.csv", wave1), ("wave_2.csv", wave2), ("wave_3.csv", wave3)]:
    with open(f"{incoming_dir}/{name}", "w") as f:
        f.write(content)

print(f"✅ Wrote 3 incoming files to {incoming_dir}/")
print("   wave_1.csv — 5 readings from SITE_DE_001 at 09:00")
print("   wave_2.csv — 5 readings from SITE_ES_001 at 09:15")
print("   wave_3.csv — 5 readings from SITE_DE_001 at 09:30")

### 작업 4.1 — Auto Loader 읽기 스트림 구성

`spark.readStream`과 `cloudFiles` 형식을 사용하여 `incoming/` 디렉토리의 새 CSV 파일을 모니터링하는 스트리밍 DataFrame을 생성합니다. Auto Loader는 자동으로 처리한 파일을 추적합니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I configure Auto Loader in PySpark using spark.readStream and the cloudFiles format to read CSV files from a Unity Catalog volume, with schemaLocation?"*

**힌트:**
- `spark.readStream`에서 `.format("cloudFiles")`를 사용합니다
- `cloudFiles.format`을 `csv`로 설정합니다
- `cloudFiles.schemaLocation`을 볼륨 내부 경로(예: `/Volumes/solaris_lab/bronze/raw_files/schema`)로 설정합니다
- `cloudFiles.inferColumnTypes`를 `true`로 설정하여 숫자 열이 올바르게 타입되도록 합니다
- `/Volumes/solaris_lab/bronze/raw_files/incoming/`에서 로드합니다

In [ ]:
df_autoloader = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/Volumes/solaris_lab/bronze/raw_files/schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .load("/Volumes/solaris_lab/bronze/raw_files/incoming/")
)

### 작업 4.2 — 스트림을 델타 테이블에 작성

Auto Loader 스트림을 `solaris_lab.bronze.solar_readings_stream`에 작성합니다. 모든 현재 사용 가능한 파일을 처리한 다음 중지하는 `trigger(availableNow=True)`를 사용합니다 — 이것이 예약된 수집 작업에 적합한 패턴입니다.

Auto Loader가 진행 상황을 추적하고 작업이 재시작되면 올바르게 재개할 수 있도록 **검사점 위치**도 지정해야 합니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How do I write an Auto Loader (spark.readStream cloudFiles) stream to a Unity Catalog table using writeStream, trigger(availableNow=True), and a checkpoint location?"*

**힌트:**
- 스트리밍 DataFrame에서 `.writeStream`을 사용합니다
- `checkpointLocation`을 `/Volumes/solaris_lab/bronze/raw_files/checkpoint`로 설정합니다
- 모든 보류 중인 파일을 처리하고 중지하려면 `.trigger(availableNow=True)`를 사용합니다
- 싱크로 `.toTable("solaris_lab.bronze.solar_readings_stream")`를 사용합니다
- 완료 대기를 위해 쿼리 객체에서 `.awaitTermination()`을 호출합니다

In [ ]:
query = (
    df_autoloader.writeStream
    .option("checkpointLocation", "/Volumes/solaris_lab/bronze/raw_files/checkpoint")
    .trigger(availableNow=True)
    .toTable("solaris_lab.bronze.solar_readings_stream")
)

query.awaitTermination()
print("✅ Auto Loader stream completed")

### 작업 4.3 — Auto Loader 수집 검증

`solaris_lab.bronze.solar_readings_stream`을 조회하여 모든 15개 행(웨이브당 5개 × 3 웨이브)이 로드되었는지 확인합니다. 그런 다음 write 스트림 셀을 두 번째로 실행하여 중복 행이 생성되지 않았는지 확인합니다 — Auto Loader의 정확히 한 번 보장을 보여줍니다.

> 🤖 **Genie Code 팁:** 요청하세요:
> *"How can I verify that Auto Loader ingested the correct number of rows and didn't create duplicates?"*

**힌트:**
- `SELECT COUNT(*) FROM solaris_lab.bronze.solar_readings_stream`를 사용합니다
- 작업 4.2의 셀을 다시 실행하고 다시 개수합니다 — 결과는 여전히 15여야 합니다

In [ ]:
%sql
-- Count all rows — should be 15 (5 rows × 3 wave files)
SELECT COUNT(*) AS total_rows FROM solaris_lab.bronze.solar_readings_stream;

In [ ]:
%sql
-- Show a sample of rows ordered by reading_timestamp
SELECT reading_id, site_id, panel_id, reading_timestamp, power_kw, status
FROM solaris_lab.bronze.solar_readings_stream
ORDER BY reading_timestamp, reading_id;

## ✅ 정리 (선택 사항)

이 랩에서 생성한 리소스를 제거하려면 아래 셀을 실행합니다.

> ⚠️ 이는 `solaris_lab` 카탈로그 및 모든 테이블과 볼륨을 포함한 모든 내용을 영구적으로 삭제합니다.

In [ ]:
%sql
-- Uncomment and run to remove all lab resources
-- DROP CATALOG IF EXISTS solaris_lab CASCADE;